# 4. Formulation variationnelle et lemme de Lax-Milgram

**Objectif** : dériver la formulation faible du problème de Poisson, vérifier numériquement
les hypothèses de Lax-Milgram (continuité et coercivité de la forme bilinéaire),
et résoudre le problème par éléments finis P1.

## Problème fort

$$-u''(x) = f(x) \text{ sur } (0,1), \qquad u(0) = u(1) = 0.$$

## Formulation faible

Trouver $u \in H^1_0(0,1)$ tel que pour tout $v \in H^1_0(0,1)$ :
$$a(u,v) = \ell(v), \qquad a(u,v) = \int_0^1 u'v'\,dx, \quad \ell(v) = \int_0^1 f v\,dx.$$

**Lax-Milgram** garantit l'existence et l'unicité car :
- $a$ est **continue** : $|a(u,v)| \le \|u\|_{H^1_0}\|v\|_{H^1_0}$
- $a$ est **coercive** : $a(u,u) = \|u'\|^2_{L^2} \ge \frac{1}{C_{\text{Poincaré}}^2}\|u\|^2_{H^1_0}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve

# ── Éléments finis P1 sur [0,1] avec N+1 noeuds ──────────────────────────────
def solve_poisson_FEM(f_func, N=50):
    """
    Résout -u'' = f sur (0,1), u(0)=u(1)=0
    par éléments finis P1 (fonctions chapeau) sur N intervalles égaux.

    Retourne (x_int, u_h) : noeuds intérieurs et solution approchée.
    """
    h   = 1.0 / N
    x   = np.linspace(0, 1, N + 1)
    # Noeuds intérieurs : x_1, ..., x_{N-1}
    x_int = x[1:N]
    n_int = N - 1

    # Matrice de rigidité A : A_{ij} = a(phi_j, phi_i) = int phi_j' phi_i' dx
    # Pour les éléments P1 uniformes : A = (1/h) * tridiag(-1, 2, -1)
    A = (1.0 / h) * (2 * np.eye(n_int)
                     - np.diag(np.ones(n_int-1), 1)
                     - np.diag(np.ones(n_int-1),-1))

    # Vecteur de charge F : F_i = int f * phi_i dx  (quadrature trapèzes)
    F = np.array([h * f_func(x_int[i]) for i in range(n_int)])

    # Résolution  A u = F
    u_int = solve(A, F)
    return x_int, u_int, A

# ── Test 1 : f = pi² sin(pi x), solution exacte u = sin(pi x) ────────────────
f1      = lambda x: np.pi**2 * np.sin(np.pi * x)
u_exact = lambda x: np.sin(np.pi * x)

x_int, u_h, A = solve_poisson_FEM(f1, N=100)
err_sup = np.max(np.abs(u_h - u_exact(x_int)))
print(f'Erreur max ||u_h - u_exact||_inf = {err_sup:.2e}  (attendu ~ h² = {(1/100)**2:.2e})')

plt.figure(figsize=(9, 4))
plt.plot(x_int, u_exact(x_int), 'k-', lw=2, label='Solution exacte $\\sin(\\pi x)$')
plt.plot(x_int, u_h, 'o--', ms=3, color='steelblue', label='FEM P1 (N=100)')
plt.title('Problème de Poisson : $-u\'\'= \\pi^2\\sin(\\pi x)$, $u(0)=u(1)=0$')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Vérification numérique des hypothèses de Lax-Milgram ─────────────────────
# On vérifie que la forme bilinéaire discrète a_h(u,v) = u^T A v
# est continue (bornes supérieures) et coercive (borne inférieure).

N   = 50
h   = 1.0 / N
n   = N - 1
A_mat = (1/h) * (2*np.eye(n) - np.diag(np.ones(n-1),1) - np.diag(np.ones(n-1),-1))

eigvals = np.linalg.eigvalsh(A_mat)

print('Valeurs propres de la matrice de rigidité A (extremes) :')
print(f'  lambda_min = {eigvals.min():.6f}  (coercivité : a(u,u) >= lambda_min * ||u||²)')
print(f'  lambda_max = {eigvals.max():.6f}  (continuité  : a(u,v) <= lambda_max * ||u|| ||v||)')
print(f'  Conditionnement kappa = {eigvals.max()/eigvals.min():.1f}  (~ 4N²/pi² = {4*N**2/np.pi**2:.1f})')

# Coercivité théorique : a(u,u) = ||u'||² >= (pi²) * ||u||²_{L2} (inégalité de Poincaré)
# car la constante de Poincaré sur [0,1] est C = 1/pi
print(f'\nVérification Poincaré : lambda_min * h >= pi² = {np.pi**2:.4f}')
print(f'  lambda_min * h = {eigvals.min() * h:.4f}  (converge vers 4 sin²(pi/(2N))*N ~ pi² pour N grand)')

In [ ]:
# ── Convergence de la méthode : ||u_h - u||_{H¹₀} en fonction de h ─────────────
Ns    = [10, 20, 50, 100, 200, 500]
errs  = []
for N in Ns:
    xi, uh, _ = solve_poisson_FEM(f1, N=N)
    # Erreur en norme H1 : approx par gradient
    duh    = np.gradient(uh, 1/N)
    du_ex  = np.pi * np.cos(np.pi * xi)
    err_H1 = np.sqrt(np.trapezoid((duh - du_ex)**2, xi))
    errs.append(err_H1)

hs = [1/N for N in Ns]
plt.figure(figsize=(6, 4))
plt.loglog(hs, errs, 'o-', color='steelblue', label=r'$\|u_h - u\|_{H^1_0}$')
plt.loglog(hs, [h for h in hs], 'k--', label='Pente 1 (ordre 1 en H¹)')
plt.xlabel('h')
plt.title('Convergence FEM P1 pour Poisson')
plt.legend(); plt.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()
print('Ordre empirique :', np.polyfit(np.log(hs), np.log(errs), 1)[0])

## Bilan

Le lemme de **Lax-Milgram** assure que si $a : V \times V \to \mathbb{R}$ est
**continue** et **coercive** sur un espace de Hilbert $V$, alors pour tout
$\ell \in V'$ il existe un unique $u \in V$ tel que $a(u,v) = \ell(v)$ pour tout $v$.

La vérification numérique confirme :
- Continuité : $\lambda_{\max} < +\infty$ ✓
- Coercivité : $\lambda_{\min} > 0$ ✓ (garanti par l'inégalité de Poincaré)
- Convergence d'ordre 1 en norme $H^1$ pour les éléments P1 ✓